# Mall Customer Segmentation using K-Means
Dataset: Kaggle Mall Customer Segmentation Dataset

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score

In [ ]:
df=pd.read_csv('../data/Mall_Customers.csv')
df.columns=df.columns.str.strip()
print(df.shape)
df.head()

In [ ]:
print(df.isna().sum())
print('Duplicates:',df.duplicated().sum())
display(df.describe(include='all').T)

In [ ]:
sns.scatterplot(data=df,x='Annual Income (k$)',y='Spending Score (1-100)',hue='Gender')
plt.title('Income vs Spending Score'); plt.show()

In [ ]:
features=['Age','Annual Income (k$)','Spending Score (1-100)']
prep=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
X=prep.fit_transform(df[features])

In [ ]:
ks=range(2,11); inertia=[]; sil=[]
for k in ks:
    m=KMeans(n_clusters=k,random_state=42,n_init=20); y=m.fit_predict(X)
    inertia.append(m.inertia_); sil.append(silhouette_score(X,y))
best_k=list(ks)[int(np.argmax(sil))]
print('Best K:',best_k)

In [ ]:
plt.plot(list(ks),inertia,marker='o'); plt.xlabel('K'); plt.ylabel('Inertia'); plt.title('Elbow Method'); plt.show()
plt.plot(list(ks),sil,marker='o'); plt.xlabel('K'); plt.ylabel('Silhouette'); plt.title('Silhouette Analysis'); plt.show()

In [ ]:
model=KMeans(n_clusters=best_k,random_state=42,n_init=50)
df['Cluster']=model.fit_predict(X)
print('Final silhouette:',round(silhouette_score(X,df['Cluster']),4))
display(df.groupby('Cluster')[features].mean().round(2))

In [ ]:
sns.scatterplot(data=df,x='Annual Income (k$)',y='Spending Score (1-100)',hue='Cluster',palette='tab10',s=80)
plt.title('Final Customer Segments'); plt.show()